# 15-2: Repaso — Logistic Regression and Count Models

**Course:** Models of Statistical Analysis (MAE) — Universidad de los Andes  
**Instructor:** Prof. Alejandra Tabares  
**Week:** 15 — Final Review

## 1. Logistic Regression — Summary

### Model

$$
\text{logit}(\pi_i) = \ln\!\left(\frac{\pi_i}{1-\pi_i}\right) = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip}
$$

$$
\pi_i = P(Y_i = 1 \mid \mathbf{x}_i) = \frac{e^{\mathbf{x}_i^\top \boldsymbol{\beta}}}{1 + e^{\mathbf{x}_i^\top \boldsymbol{\beta}}} = \frac{1}{1 + e^{-\mathbf{x}_i^\top \boldsymbol{\beta}}}
$$

### Why Not OLS for Binary Outcomes?

Linear probability model (LPM) with OLS can predict $\hat{\pi} < 0$ or $\hat{\pi} > 1$, and the error variance is heteroscedastic. The logit link constrains predictions to $(0,1)$.

### Interpretation — Three Scales

| Scale | Expression | Interpretation of $\beta_j$ |
|---|---|---|
| **Log-odds** (logit) | $\mathbf{x}^\top\boldsymbol{\beta}$ | Additive: $\beta_j$ = change in log-odds per unit $x_j$ |
| **Odds ratio** | $e^{\beta_j}$ | Multiplicative: OR = factor by which odds changes per unit $x_j$ |
| **Probability** | $1/(1+e^{-\mathbf{x}^\top\boldsymbol{\beta}})$ | Non-linear; depends on values of all $x$'s |

### Estimation — Maximum Likelihood

The log-likelihood for $n$ binary observations:

$$
\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \left[ y_i \ln \pi_i + (1-y_i) \ln(1-\pi_i) \right]
$$

No closed-form solution; solved iteratively via **IRLS** (Iteratively Reweighted Least Squares).

### Model Evaluation

| Metric | What it measures |
|---|---|
| **Deviance** | $-2\ell(\hat{\boldsymbol{\beta}})$; lower is better |
| **AIC** | $-2\ell + 2p$ |
| **McFadden $R^2$** | $1 - \ell(\hat{\boldsymbol{\beta}})/\ell(\boldsymbol{\beta}_0)$ |
| **Confusion matrix** | Accuracy, sensitivity (recall), specificity, PPV |
| **AUC-ROC** | Probability that model ranks a random positive above a random negative; 0.5 = random, 1.0 = perfect |

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2025)
n = 400

# Simulate binary outcome: disease = 1
age     = np.random.normal(55, 12, n)
bmi     = np.random.normal(27, 5, n)
smoking = np.random.binomial(1, 0.3, n)

# True logit: log-odds = -6 + 0.06*age + 0.08*bmi + 1.2*smoking
logit_true = -6 + 0.06 * age + 0.08 * bmi + 1.2 * smoking
pi_true    = 1 / (1 + np.exp(-logit_true))
disease    = np.random.binomial(1, pi_true, n)

df = pd.DataFrame({'disease': disease, 'age': age, 'bmi': bmi, 'smoking': smoking})

print('Simulated dataset: disease ~ age + bmi + smoking')
print(f'Outcome prevalence: {disease.mean():.3f} ({disease.sum()} cases out of {n})')
print(df.head(8).round(2).to_string(index=False))

In [ ]:
# ── Fit logistic regression ───────────────────────────────────────────────────

logit_model = smf.logit('disease ~ age + bmi + smoking', data=df).fit()
print(logit_model.summary())

# Odds ratios with 95% CI
or_table = pd.DataFrame({
    'OR':       np.exp(logit_model.params),
    'OR_lower': np.exp(logit_model.conf_int()[0]),
    'OR_upper': np.exp(logit_model.conf_int()[1]),
    'p-value':  logit_model.pvalues
}).round(4)

print('\nOdds Ratio Table (95% CI)')
print('='*55)
print(or_table.to_string())

print('\nInterpretation example:')
or_smoking = np.exp(logit_model.params['smoking'])
print(f'  Smoking OR = {or_smoking:.3f}: smokers have {or_smoking:.2f}x the odds')
print('  of disease compared to non-smokers, holding age and BMI constant.')

In [ ]:
# ── Confusion matrix and ROC curve ───────────────────────────────────────────

y_pred_prob = logit_model.predict(df)
threshold   = 0.5
y_pred      = (y_pred_prob >= threshold).astype(int)
y_true      = df['disease'].values

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
accuracy    = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn)  # recall
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp)  # precision
auc         = roc_auc_score(y_true, y_pred_prob)

print('Confusion Matrix Summary (threshold = 0.5)')
print('='*45)
print(f'  True Positives  (TP): {tp}')
print(f'  True Negatives  (TN): {tn}')
print(f'  False Positives (FP): {fp}')
print(f'  False Negatives (FN): {fn}')
print(f'  Accuracy:             {accuracy:.3f}')
print(f'  Sensitivity (Recall): {sensitivity:.3f}')
print(f'  Specificity:          {specificity:.3f}')
print(f'  PPV (Precision):      {ppv:.3f}')
print(f'  AUC-ROC:              {auc:.3f}')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix heatmap
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No disease', 'Disease'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix (threshold={threshold})', fontsize=12)

# ROC curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2,
             label=f'Logistic Regression (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

## 2. Poisson Regression — Summary

### Model

$$
Y_i \sim \text{Poisson}(\mu_i), \qquad \ln(\mu_i) = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip}
$$

### Interpretation — Rate Ratios

$$
e^{\hat{\beta}_j} = \text{Rate Ratio (RR)}
$$

For a one-unit increase in $x_j$, the expected count is **multiplied by $e^{\hat{\beta}_j}$**, holding other predictors fixed.

### Key Assumptions

1. **Log-linearity**: $\ln(\mu)$ is linear in the predictors
2. **Independence**: observations are independent
3. **Equidispersion**: $\text{Var}(Y_i) = E[Y_i] = \mu_i$ (mean = variance)

### Offset

When modeling **rates** (counts per unit of exposure $t_i$):

$$
\ln(\mu_i / t_i) = \mathbf{x}_i^\top \boldsymbol{\beta} \quad \Longleftrightarrow \quad \ln(\mu_i) = \ln(t_i) + \mathbf{x}_i^\top \boldsymbol{\beta}
$$

The term $\ln(t_i)$ is included as an **offset** with coefficient fixed at 1.

### Overdispersion

If $\hat{D}/(n-p-1) \gg 1$ (say, $> 1.5$), the variance exceeds the mean — **overdispersion**. This inflates test statistics and produces overconfident confidence intervals.

**Remedies:**

| Approach | Model | When |
|---|---|---|
| Scale standard errors | Quasi-Poisson | Mild overdispersion |
| Explicit extra variance | Negative Binomial | Moderate-severe overdispersion |
| Check for zero inflation | Zero-inflated Poisson / ZINB | Excess zeros |

In [ ]:
# ── Simulate count data and fit Poisson + Negative Binomial ──────────────────

np.random.seed(42)
n_cnt = 300

x1 = np.random.normal(0, 1, n_cnt)
x2 = np.random.binomial(1, 0.4, n_cnt)      # binary covariate
mu_pois = np.exp(1.5 + 0.6 * x1 + 0.8 * x2)

# Overdispersed counts: use Negative Binomial (dispersion r = 3)
r_nb  = 3.0
p_nb  = r_nb / (r_nb + mu_pois)
y_nb  = np.random.negative_binomial(r_nb, p_nb, n_cnt)

# Also generate properly Poisson-distributed counts for comparison
y_pois = np.random.poisson(mu_pois, n_cnt)

df_cnt = pd.DataFrame({'y_nb': y_nb, 'y_pois': y_pois, 'x1': x1, 'x2': x2})

print('Count dataset summary')
print(f'  y_nb   — mean: {y_nb.mean():.2f}, var: {y_nb.var():.2f},',
      f'var/mean = {y_nb.var()/y_nb.mean():.2f}  (overdispersed if >> 1)')
print(f'  y_pois — mean: {y_pois.mean():.2f}, var: {y_pois.var():.2f},',
      f'var/mean = {y_pois.var()/y_pois.mean():.2f}  (equidispersed ≈ 1)')

# Poisson on overdispersed data
pois_nb  = smf.glm('y_nb ~ x1 + x2', data=df_cnt,
                    family=sm.families.Poisson()).fit()
# Negative Binomial on overdispersed data
nb_model = smf.glm('y_nb ~ x1 + x2', data=df_cnt,
                    family=sm.families.NegativeBinomial()).fit()
# Poisson on properly distributed data
pois_ok  = smf.glm('y_pois ~ x1 + x2', data=df_cnt,
                    family=sm.families.Poisson()).fit()

print('\n' + '='*65)
print('Poisson GLM on OVERDISPERSED counts (y_nb)')
print('='*65)
print(pois_nb.summary2().tables[1].to_string())
disp_ratio = pois_nb.deviance / pois_nb.df_resid
print(f'\n  Residual deviance / df = {disp_ratio:.2f}',
      '→ OVERDISPERSED (should be ~1)' if disp_ratio > 1.5 else '→ OK')

print('\n' + '='*65)
print('Negative Binomial GLM on OVERDISPERSED counts (y_nb)')
print('='*65)
print(nb_model.summary2().tables[1].to_string())

print('\n' + '='*65)
print('Poisson GLM on EQUIDISPERSED counts (y_pois)')
print('='*65)
print(pois_ok.summary2().tables[1].to_string())
disp_ratio_ok = pois_ok.deviance / pois_ok.df_resid
print(f'\n  Residual deviance / df = {disp_ratio_ok:.2f}',
      '→ OVERDISPERSED' if disp_ratio_ok > 1.5 else '→ OK (equidispersed)')

In [ ]:
# ── AIC comparison: Poisson vs NB on overdispersed data ──────────────────────

aic_compare = pd.DataFrame({
    'Model':        ['Poisson (on overdispersed)', 'Negative Binomial'],
    'AIC':          [pois_nb.aic, nb_model.aic],
    'Deviance':     [pois_nb.deviance, nb_model.deviance],
    'Dev/df':       [pois_nb.deviance/pois_nb.df_resid,
                     nb_model.deviance/nb_model.df_resid]
}).round(3)

print('AIC Comparison: Poisson vs Negative Binomial')
print('='*60)
print(aic_compare.to_string(index=False))
print('\n  Lower AIC = better fit. NB should win when data are overdispersed.')

# Rate ratio table for NB model
rr_table = pd.DataFrame({
    'RR':       np.exp(nb_model.params),
    'RR_lower': np.exp(nb_model.conf_int()[0]),
    'RR_upper': np.exp(nb_model.conf_int()[1]),
    'p-value':  nb_model.pvalues
}).round(4)
print('\nNegative Binomial — Rate Ratio Table (95% CI)')
print('='*55)
print(rr_table.to_string())

## 3. Comparison: OLS vs Logistic vs Poisson

| Feature | **OLS (Linear)** | **Logistic Regression** | **Poisson Regression** |
|---|---|---|---|
| **Response type** | Continuous, unbounded | Binary (0/1) or proportion | Non-negative integer (count) |
| **Distribution** | Normal ($\mathcal{N}$) | Binomial | Poisson |
| **Link function** | Identity: $\mu = \mathbf{x}^\top\boldsymbol{\beta}$ | Logit: $\ln(\pi/(1-\pi))$ | Log: $\ln(\mu)$ |
| **Coefficient interpretation** | Additive change in $E[Y]$ | Additive change in log-odds; $e^\beta$ = OR | Additive change in $\ln(\mu)$; $e^\beta$ = RR |
| **Estimation** | OLS (closed form) | MLE (IRLS) | MLE (IRLS) |
| **Key assumption** | Homoscedastic normal errors | $Y_i \sim \text{Bernoulli}(\pi_i)$ | $\text{Var}(Y_i) = \mu_i$ (equidispersion) |
| **Goodness-of-fit** | $R^2$, RMSE, F-test | Deviance, AUC, McFadden $R^2$ | Deviance, deviance / df, AIC |
| **Overdispersion** | Not applicable (estimates $\sigma^2$) | Can use quasi-binomial | Use NB or quasi-Poisson |
| **Prediction** | $\hat{y} \in (-\infty, +\infty)$ | $\hat{\pi} \in (0, 1)$ | $\hat{\mu} \in (0, +\infty)$ |

### When to Use Each Model

| Situation | Model |
|---|---|
| Predict exam score from study hours | OLS |
| Predict whether a patient has a disease (yes/no) | Logistic |
| Predict number of hospital admissions per day | Poisson |
| Count data with variance $\gg$ mean | Negative Binomial |
| Predict house price (positive, skewed) | Gamma GLM or log-transform OLS |
| Time until event (survival) | Cox PH or Kaplan-Meier |